# Preprocessing Steps

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import h5py
import sys
import os
import re 

sys.path.insert(0, os.path.abspath("."))
from eeg_functions.load_and_plot import *

# Normalise Channel Name and Create New Dataset

> Normalise channel name and drop the extra one

In [ ]:
def normalize(ch):
    if isinstance(ch, (bytes, bytearray)):
        ch = ch.decode()

    ch = ch.upper().strip().replace(" ", "")
    ch = re.sub(r"-\d+$", "", ch)

    return ch


def drop_duplicates(signals, ch_names):

    seen = {}
    keep_idx = []

    for i, ch in enumerate(ch_names):

        ch_norm = normalize(ch)

        # Keep first occurence only
        if ch_norm not in seen:
            seen[ch_norm] = i
            keep_idx.append(i)

    return signals[keep_idx, :], [normalize(ch_names[i]) for i in keep_idx]

In [ ]:

def process_and_save(path, target_channels, clean_dir):

    with h5py.File(path, "r") as f:
        signals = f["signals"][:]
        fs = f["sampling_rate"][()]
        ch_names = f["channel_names"][:]
        intervals = f["seizure_intervals"][:] if "seizure_intervals" in f else np.array([])

    # decode
    ch_names = [
        c.decode() if isinstance(c, (bytes, bytearray)) else c
        for c in ch_names
    ]

    # normalize
    ch_names = [normalize(c) for c in ch_names]

    # drop duplicates (your decision)
    signals, ch_names = drop_duplicates(signals, ch_names)

    # enforce ordering + filter to target set
    index = {ch: i for i, ch in enumerate(ch_names)}

    missing = [c for c in target_channels if c not in index]
    if missing:
        print(f"Skipping {path} (missing: {missing})")
        return

    idx = [index[c] for c in target_channels]

    signals = signals[idx, :]
    ch_names = target_channels

    # save to new folder
    filename = os.path.basename(path)
    out_path = os.path.join(clean_dir, filename)

    with h5py.File(out_path, "w") as f:
        f.create_dataset("signals", data=signals, compression="gzip")
        f.create_dataset("sampling_rate", data=fs)
        f.create_dataset("channel_names", data=np.array(ch_names, dtype="S"))
        f.create_dataset("seizure_intervals", data=intervals)

    print(f"Saved → {out_path}")

In [ ]:
RAW_DIR = "../Dataset_hdf5"
CLEAN_DIR = "../Dataset_hdf5_clean"
os.makedirs(CLEAN_DIR, exist_ok=True)

> These are the 22 most common channels

In [ ]:
target_channels = [
    'FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1',
    'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1',
    'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2',
    'FP2-F8', 'F8-T8', 'T8-P8', 'P8-O2',
    'FZ-CZ', 'CZ-PZ', 'P7-T7', 'T7-FT9',
    'FT9-FT10', 'FT10-T8'
]

In [86]:
for root, _, files in os.walk(RAW_DIR):
    if "chb24" not in root:  # ← add this check
        continue
    for file in files:
        if file.endswith(".h5"):
            path = os.path.join(root, file)
            process_and_save(
                path,
                target_channels,
                CLEAN_DIR
            )

Saved → ../Dataset_hdf5_clean\chb24_01.h5
Saved → ../Dataset_hdf5_clean\chb24_02.h5
Saved → ../Dataset_hdf5_clean\chb24_03.h5
Saved → ../Dataset_hdf5_clean\chb24_04.h5
Saved → ../Dataset_hdf5_clean\chb24_05.h5
Saved → ../Dataset_hdf5_clean\chb24_06.h5
Saved → ../Dataset_hdf5_clean\chb24_07.h5
Saved → ../Dataset_hdf5_clean\chb24_08.h5
Saved → ../Dataset_hdf5_clean\chb24_09.h5
Saved → ../Dataset_hdf5_clean\chb24_10.h5
Saved → ../Dataset_hdf5_clean\chb24_11.h5
Saved → ../Dataset_hdf5_clean\chb24_12.h5
Saved → ../Dataset_hdf5_clean\chb24_13.h5
Saved → ../Dataset_hdf5_clean\chb24_14.h5
Saved → ../Dataset_hdf5_clean\chb24_15.h5
Saved → ../Dataset_hdf5_clean\chb24_16.h5
Saved → ../Dataset_hdf5_clean\chb24_17.h5
Saved → ../Dataset_hdf5_clean\chb24_18.h5
Saved → ../Dataset_hdf5_clean\chb24_19.h5
Saved → ../Dataset_hdf5_clean\chb24_20.h5
Saved → ../Dataset_hdf5_clean\chb24_21.h5
Saved → ../Dataset_hdf5_clean\chb24_22.h5


In [ ]:
for root, _, files in os.walk(RAW_DIR):
    # if "chb13" not in root: 
    #     continue
    for file in files:
        if file.endswith(".h5"):

            path = os.path.join(root, file)

            process_and_save(
                path,
                target_channels,
                CLEAN_DIR
            )

Saved → ../Dataset_hdf5_clean\chb01_01.h5
Saved → ../Dataset_hdf5_clean\chb01_02.h5
Saved → ../Dataset_hdf5_clean\chb01_03.h5
Saved → ../Dataset_hdf5_clean\chb01_04.h5
Saved → ../Dataset_hdf5_clean\chb01_05.h5
Saved → ../Dataset_hdf5_clean\chb01_06.h5
Saved → ../Dataset_hdf5_clean\chb01_07.h5
Saved → ../Dataset_hdf5_clean\chb01_08.h5
Saved → ../Dataset_hdf5_clean\chb01_09.h5
Saved → ../Dataset_hdf5_clean\chb01_10.h5
Saved → ../Dataset_hdf5_clean\chb01_11.h5
Saved → ../Dataset_hdf5_clean\chb01_12.h5
Saved → ../Dataset_hdf5_clean\chb01_13.h5
Saved → ../Dataset_hdf5_clean\chb01_14.h5
Saved → ../Dataset_hdf5_clean\chb01_15.h5
Saved → ../Dataset_hdf5_clean\chb01_16.h5
Saved → ../Dataset_hdf5_clean\chb01_17.h5
Saved → ../Dataset_hdf5_clean\chb01_18.h5
Saved → ../Dataset_hdf5_clean\chb01_19.h5
Saved → ../Dataset_hdf5_clean\chb01_20.h5
Saved → ../Dataset_hdf5_clean\chb01_21.h5
Saved → ../Dataset_hdf5_clean\chb01_22.h5
Saved → ../Dataset_hdf5_clean\chb01_23.h5
Saved → ../Dataset_hdf5_clean\chb0

# Check File Number

In [87]:
root = "../Dataset_hdf5"
count = 0

for _, _, files in os.walk(root):
    for file in files:
        if file.endswith(".h5"):
            count += 1

print("Total HDF5 files:", count)

Total HDF5 files: 686


In [88]:
root = "../Dataset_hdf5_clean"
count = 0

for _, _, files in os.walk(root):
    for file in files:
        if file.endswith(".h5"):
            count += 1

print("Total HDF5 files:", count)

Total HDF5 files: 655


## Compare with txt file

In [81]:
from pathlib import Path

# ------------------------------------------
# File containing your expected EDF paths
# ------------------------------------------
expected_txt = "RECORDS.txt"

# Root folder of your HDF5 dataset
root = Path("../Dataset_hdf5")

# ------------------------------------------
# Read expected EDF paths
# ------------------------------------------
with open(expected_txt, "r") as f:
    expected = {
        line.strip().replace(".edf", ".h5")
        for line in f
        if line.strip()
    }

# ------------------------------------------
# Find all actual H5 files recursively
# ------------------------------------------
actual = {
    str(p.relative_to(root)).replace("\\", "/")
    for p in root.rglob("*.h5")
}

# ------------------------------------------
# Compare
# ------------------------------------------
missing = sorted(expected - actual)
extra = sorted(actual - expected)

print("=" * 60)
print(f"Expected : {len(expected)}")
print(f"Found    : {len(actual)}")
print(f"Missing  : {len(missing)}")
print(f"Extra    : {len(extra)}")
print("=" * 60)

if missing:
    print("\nMISSING FILES:")
    for f in missing:
        print(f)

if extra:
    print("\nEXTRA FILES:")
    for f in extra:
        print(f)

Expected : 686
Found    : 686
Missing  : 0
Extra    : 0
